<a href="https://colab.research.google.com/github/AjayLohith/Naive-RAG/blob/main/Naive_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#BASIC RAG APP


In [43]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


###Imports

In [44]:
!pip install openai chromadb python-dotenv

In [45]:
import os
import dotenv
from openai import OpenAI, api_key
from  dotenv import load_dotenv
import json
import chromadb
from openai.types.responses import responses_client_event
from websockets import client

In [46]:
from google.colab import userdata
groq_api_key=userdata.get('GROQ_API_KEY')

## Interacting with LLM

In [47]:
from openai import OpenAI

client=OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

In [48]:
response=client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role":"system","content":"You are a professional RAG expert in Agentic AI field"},
        {"role":"user","content":"Explainn me types of rags in 5 bullet points"}
    ],
    temperature=0
)
print(response.choices[0].message.content)

As a RAG (Retrieve, Augment, Generate) expert in the Agentic AI field, I'd be happy to explain the types of RAGs. Here are 5 key types:

* **Retrieve-only RAG**: This type of RAG focuses solely on retrieving relevant information from a knowledge base or database, without generating new text. It's often used for question-answering tasks or providing definitions.
* **Augment-only RAG**: In this type, the RAG augments existing text by adding, modifying, or deleting content to improve its relevance, accuracy, or coherence. This is useful for tasks like text editing, summarization, or data enrichment.
* **Generate-only RAG**: This type of RAG generates new text from scratch, without relying on existing content. It's commonly used for creative writing, content generation, or chatbot responses.
* **Retrieve-and-Generate (RAG) RAG**: This type combines the retrieve and generate capabilities, where the RAG retrieves relevant information and then generates new text based on that information. Thi

## Loading and chunking data from document

In [49]:
with open("/content/drive/MyDrive/Naive RAG/_data/company_hr_policy.txt","r")as f:
  hr_doc=f.read()

with open("/content/drive/MyDrive/Naive RAG/_data/engineering_standards.txt","r")as f:
  engineering_standards=f.read()

with open("/content/drive/MyDrive/Naive RAG/_data/onboarding_guide.txt","r")as f:
  onboarding_guide=f.read()

with open("/content/drive/MyDrive/Naive RAG/_data/product_knowledge_base.txt","r")as f:
  product_knowledge=f.read()

with open("/content/drive/MyDrive/Naive RAG/_data/security_policy.txt","r")as f:
  security_policy=f.read()

print(hr_doc)
# print(engineering_standards)
# print(onboarding_guide)
# print(product_knowledge)
# print(security_policy)


NovaTech Solutions — Employee Handbook & HR Policy
Version 3.2 | Last Updated: January 2026

SECTION 1: LEAVE POLICY

Annual Leave:
All full-time employees are entitled to 24 days of paid annual leave per calendar year. Leave accrues at the rate of 2 days per month. New employees can start using accrued leave after completing 3 months of service. Unused leave up to 10 days can be carried forward to the next year. Any leave beyond 10 days will lapse on December 31st.

Sick Leave:
Employees are entitled to 12 days of sick leave per year. Sick leave for more than 3 consecutive days requires a medical certificate from a registered medical practitioner. Sick leave cannot be carried forward or encashed. In case of extended illness beyond 12 days, employees may apply for medical leave without pay, subject to HR approval.

Casual Leave:
Employees are entitled to 6 days of casual leave per year. Casual leave cannot be taken for more than 3 consecutive days. Prior approval from the reporting man

In [50]:
print(f"HR Document lenght: {len(hr_doc)} chars")
print(f"HR Document lenght: {len(engineering_standards)} chars")

print(f"HR document words {len(hr_doc.split())}")

HR Document lenght: 7464 chars
HR Document lenght: 4941 chars
HR document words 1052


## Chunking Strategy

In [51]:
def chunk_documents(text,source_name):
  paragraph=text.strip().split("\n" "\n")
  chunks=[]

  for para in paragraph:
    para=para.strip()
    if len(para)<50:
      continue

    if para.startswith("=="):
      continue

    chunks.append({"text":para,"source":source_name})


  return chunks

In [52]:
hr_chunk=chunk_documents(hr_doc,"HR Policy")
engineering_standards_chunk=chunk_documents(engineering_standards,"Engineering Standards")
onboarding_guide_chunk=chunk_documents(onboarding_guide,"Onboarding Guide")
product_knowledge_chunk=chunk_documents(product_knowledge,"Product Knowledge Base")
security_policy_chunk=chunk_documents(security_policy,"Security Policy")

In [53]:
total_chunks=hr_chunk+engineering_standards_chunk+onboarding_guide_chunk+product_knowledge_chunk+security_policy_chunk
print(f"Total Chunks: {len(total_chunks)}")

Total Chunks: 120


## Storing chunks into ChromDb

In [54]:
chroma_client=chromadb.Client()
collection=chroma_client.create_collection(name="company_chunks")

InternalError: Collection [company_chunks] already exists

In [55]:
documents=[]
ids=[]
metadata=[]

for i,chunk in enumerate(total_chunks):
  documents.append(chunk['text'])
  ids.append(f"chunk_{i}")
  metadata.append({"source": chunk["source"]})

print(documents[0])
print(ids[0])
print(metadata[0])

NovaTech Solutions — Employee Handbook & HR Policy
Version 3.2 | Last Updated: January 2026
chunk_0
{'source': 'HR Policy'}


##### Here Chromadb automatically process embeddings using sentence-transformer

In [56]:
collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadata
)

In [57]:
print(f"Stored documents in ChromaDb :{len(documents)}")
# print(f"Stored documents in ChromaDb :{documents}")

Stored documents in ChromaDb :120


#### Behind the scenes of similarity seach

In [58]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

sentence1="I'm driving my Rolls-Royce Phantom on the highway at 70 mph."
sentence2="I'm eating grilled chicken with rice after finishing my workout."
sentence3="Im eating on chicken"

embed1=model.encode(sentence1)
embed2=model.encode(sentence2)
embed3=model.encode(sentence3)

print(embed1.shape)
print(embed2.shape)
print(embed3.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(384,)
(384,)
(384,)


In [59]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

similarity1=cosine_similarity([embed1],[embed2])
similarity2=cosine_similarity([embed2],[embed3])
similarity3=cosine_similarity([embed1],[embed3])

print(similarity1)
print(similarity2)
print(similarity3)


[[0.19326067]]
[[0.6062665]]
[[0.12841894]]


## Construction of Retrievel Pipeline

In [60]:
def retrieve(question,n_results=3):
  results=collection.query(
      query_texts=[question],
      n_results=n_results
  )
  return results['documents'][0],results['metadatas'][0]

In [61]:
chunks , sources = retrieve("What is the work from home policy?",5)

for i in range(len(chunks)):
  print(f"----Chunk {i+1}----")
  print(f"source:{sources[i]}")
  print(f"Texts: {chunks[i]}")
  print()

----Chunk 1----
source:{'source': 'HR Policy'}
Texts: Eligibility:
All employees who have completed their probation period (6 months) are eligible for Work From Home (WFH) arrangements. Employees in their probation period may request WFH only in exceptional circumstances with manager and HR approval.

----Chunk 2----
source:{'source': 'HR Policy'}
Texts: NovaTech Solutions — Employee Handbook & HR Policy
Version 3.2 | Last Updated: January 2026

----Chunk 3----
source:{'source': 'HR Policy'}
Texts: Regular WFH:
Employees may work from home up to 2 days per week. The preferred WFH days are Wednesday and Friday, though teams may adjust based on project needs. Employees must be available during core working hours (10:00 AM to 6:00 PM IST) on WFH days.

----Chunk 4----
source:{'source': 'Security Policy'}
Texts: Internet Usage:
- Company internet is primarily for work purposes
- Limited personal use is acceptable during breaks
- The following are strictly prohibited: downloading copyrighte

# Testing RAG Pipeline

In [62]:
def ask_rag(question,n_results=3,verbose=True):
  chunks,sources=retrieve(question,n_results)

  if verbose:
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"\n{'-'*60}")
    print(f"Retrieved {len(chunks)} chunks:")

    for i,(chunks,sources) in enumerate(zip(chunks,sources)):
      print(f"[{sources['source']}] {chunks[:80]}...")
    print(f"\n{'-'*60}")

  context="\n \n".join(chunks)

  messages=[
      {
        "role":"system",
        "content":

        "You are a helpful assistance that asnwers quesitosn based ONLY on the provided context"
        "If the context does not contain informaiton to answer this quesitons, "
        "say I dont have enough context or information to answer this question"
        "Do not make up information or assume anything. Strictly answer only from the providede context"
    },
    {
        "role":"user",
        "content":f"Context: {context}\n \n---\n\nQuestion:{question}"
    }

  ]

  response=client.chat.completions.create(
      model=MODEL,
      messages=messages,
      temperature=0.1
  )
  answer=(response.choices[0].message.content)

  if verbose:
    print(f"Answer: {answer}")
    print(f"{'=' *60}")

  return answer
print("RAG pipeline is built")




RAG pipeline is built


In [63]:
ask_rag("How many days of annual leave do employees get?Just give me number no extra info")


Question: How many days of annual leave do employees get?Just give me number no extra info

------------------------------------------------------------
Retrieved 3 chunks:
[HR Policy] Annual Leave:
All full-time employees are entitled to 24 days of paid annual lea...
[HR Policy] Sick Leave:
Employees are entitled to 12 days of sick leave per year. Sick leave...
[HR Policy] Casual Leave:
Employees are entitled to 6 days of casual leave per year. Casual ...

------------------------------------------------------------
Answer: 6


'6'

In [64]:
ask_rag("What is the work from Home Policy?")


Question: What is the work from Home Policy?

------------------------------------------------------------
Retrieved 3 chunks:
[HR Policy] Eligibility:
All employees who have completed their probation period (6 months) ...
[HR Policy] NovaTech Solutions — Employee Handbook & HR Policy
Version 3.2 | Last Updated: J...
[HR Policy] Regular WFH:
Employees may work from home up to 2 days per week. The preferred W...

------------------------------------------------------------
Answer: The work from home (WFH) policy allows employees to work from home up to 2 days per week. The preferred WFH days are Wednesday and Friday, though teams may adjust based on project needs. Employees must be available during core working hours (10:00 AM to 6:00 PM) on WFH days.


'The work from home (WFH) policy allows employees to work from home up to 2 days per week. The preferred WFH days are Wednesday and Friday, though teams may adjust based on project needs. Employees must be available during core working hours (10:00 AM to 6:00 PM) on WFH days.'

In [ ]:
ask_rag("What is the capital of Germany?")

In [ ]:
ask_rag("What happens during the probation period?")

In [ ]:
# Product KB questions
ask_rag("What are the pricing plans for CloudDesk Pro?")

In [ ]:
ask_rag("How do I cancel my subscription?")

In [ ]:
ask_rag("What is the refund policy?")

In [ ]:
ask_rag("Home Policy?")

# Agentic RAG

## Creating RAG tool

In [65]:
def search_docs(query:str)->str:
  results=collection.query(
      query_texts=[query],
      n_results=3
  )
  chunks=results['documents'][0]
  return "\n \n".join(chunks)

rag_tools=[
    {
        "type":"function",
        "function":{
            "name":"search_docs",
            "description":"Searches company internal documents for information.",
            "parameters":{
                "type":"object",
                "properties":{
                    "query":{
                        "type":"string",
                        "description":"The search query."
                    }
                },
                "required":["query"]
            }
          }
      }

]

available_tools={
    "search_docs":search_docs
}

print("RAG tool defined")

RAG tool defined


In [66]:
def rag_agent(question,verbose=True):
  messages=[
      {
          "role":"system",
          "content":(
              "You are a helpful company assistance and you have access to internal documents "
              "including-HR polices and Porduct, engineering , onboarding, security"
              "Use the search_docs tool to find answers from the company documents"
              "If the documents dont contian the answer, say so clearly."
              "Always base your answers on the rerieved documets if you decided to use the tool."
              "If tool use is not needed, specifcy that you are not using tool and asnwer the quesiton based on your knowledge."
          )
      },
      {
          "role":"user",
          "content":question
      }
  ]

  if verbose:
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"\n{'='*60}")

  max_steps=3
  for step in range(max_steps):
    response=client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=rag_tools
    )

    choice=response.choices[0]

    if choice.finish_reason=="stop":
      if verbose:
        print(f"Answer: {choice.message.content}")
        print(f"{'='*60}")
      return choice.message.content

    if choice.message.tool_calls:
      messages.append(choice.message)

      for tool_call in choice.message.tool_calls:
        func_name=tool_call.function.name
        args=json.loads(tool_call.function.arguments)

        if verbose:
            print(f"Searching: \"{args['query']}\"")

        result=available_tools[func_name](**args)

        if verbose:
          print(f"Result: {len(result)}")

        messages.append(
            {
                "role":"tool",
                "tool_call_id":tool_call.id,
                "content":result
            }
        )
  return "Could not find answer within step limit."


In [67]:
rag_agent("How many sick leave days do i get per year ")


Question: How many sick leave days do i get per year 

Searching: "sick leave days per year"
Result: 963
Answer: You are entitled to 12 days of sick leave per year.


'You are entitled to 12 days of sick leave per year.'

In [68]:
rag_agent("What is Capital of Switzerland")


Question: What is Capital of Switherland

Answer: I am not using the tool as the question does not require searching company internal documents. The capital of Switzerland is Bern.


'I am not using the tool as the question does not require searching company internal documents. The capital of Switzerland is Bern.'